# Metric-only SSP benchmark

In [1]:
from pathlib import Path
import pickle
import numpy as np
import pandas as pd

base = Path('metrics_AGWP_wh_ModC_CC_T')
outdir = Path('metric_dpRF_dpAGWP_benchmark_perct')
scenarios = [('SSP1-1.9', 'ssp119'), ('SSP2-4.5', 'ssp245'), ('SSP5-8.5', 'ssp585')]
model_years = [2030, 2040, 2050]
horizons = [10, 20, 50, 100, 200]
gases = ['CO2', 'CH4', 'N2O']
metric_specs_final = {
    ('CO2', 'dpRF'): 'rf',
    ('CO2', 'dpAGWP'): 'agwp',
    ('CH4', 'dpRF'): 'rf_final',
    ('CH4', 'dpAGWP'): 'agwp_final',
    ('N2O', 'dpRF'): 'rf_final',
    ('N2O', 'dpAGWP'): 'agwp_final',
}

def flow_label(gas):
    return {'CO2': 'CO2', 'CH4': 'CH4', 'N2O': 'N2O'}[gas]

def load_obj(scenario_code, model_year):
    path = base / f'metrics_by_pIRF_tstep1ALL_{scenario_code}_fair_start1750MY{model_year}.pkl'
    with path.open('rb') as f:
        return pickle.load(f)

def get_value(obj, gas, metric_key, horizon):
    arr = np.asarray(obj['metrics'][gas][metric_key], dtype=float)
    h_arr = np.asarray(obj['meta']['H'], dtype=float)
    idx = int(np.where(h_arr == horizon)[0][0])
    return float(np.nanmean(arr[idx, :])) if arr.ndim == 2 else float(arr[idx])




In [2]:
rows = []
for model_year in model_years:
    cache = {(code, model_year): load_obj(code, model_year) for _, code in scenarios}
    for metric in ['dpRF', 'dpAGWP']:
        for gas in gases:
            metric_key = metric_specs_final[(gas, metric)]
            for horizon in horizons:
                vals = {
                    label: get_value(cache[(code, model_year)], gas, metric_key, horizon)
                    for label, code in scenarios
                }
                s1 = vals['SSP1-1.9']
                s2 = vals['SSP2-4.5']
                s5 = vals['SSP5-8.5']
                rows.append({
                    'Metric': metric,
                    'Metric source': metric_key,
                    'ModelYear': model_year,
                    'Horizon H': horizon,
                    'GHG': flow_label(gas),
                    'SSP1-1.9 value': s1,
                    'SSP2-4.5 value': s2,
                    'SSP5-8.5 value': s5,
                    'SSP2-4.5 - SSP1-1.9': s2 - s1,
                    'Difference% SSP2-4.5 vs SSP1-1.9': (s2 - s1) / s1 * 100,
                    'SSP5-8.5 - SSP1-1.9': s5 - s1,
                    'Difference% SSP5-8.5 vs SSP1-1.9': (s5 - s1) / s1 * 100,
                })

df = pd.DataFrame(rows)
df

,Metric,Metric source,ModelYear,Horizon H,GHG,SSP1-1.9 value,SSP2-4.5 value,SSP5-8.5 value,SSP2-4.5 - SSP1-1.9,Difference% SSP2-4.5 vs SSP1-1.9,SSP5-8.5 - SSP1-1.9,Difference% SSP5-8.5 vs SSP1-1.9
0,dpRF,rf,2030,10,CO2,1.165105e-15,1.171902e-15,1.175199e-15,6.797345e-18,0.583411,1.009449e-17,0.866402
1,dpRF,rf,2030,20,CO2,1.039474e-15,1.084167e-15,1.116069e-15,4.469247e-17,4.299526,7.659463e-17,7.368593
2,dpRF,rf,2030,50,CO2,8.579884e-16,1.001841e-15,1.168452e-15,1.438524e-16,16.766236,3.104636e-16,36.185051
3,dpRF,rf,2030,100,CO2,7.125229e-16,9.405829e-16,1.367921e-15,2.280600e-16,32.007391,6.553984e-16,91.982788
4,dpRF,rf,2030,200,CO2,6.009008e-16,8.837166e-16,1.524249e-15,2.828158e-16,47.065299,9.233482e-16,153.660664
...,...,...,...,...,...,...,...,...,...,...,...,...
85,dpAGWP,agwp_final,2050,10,N2O,2.567842e-12,2.635260e-12,2.665541e-12,6.741878e-14,2.625504,9.769923e-14,3.804722
86,dpAGWP,agwp_final,2050,20,N2O,4.866436e-12,4.975502e-12,5.028868e-12,1.090660e-13,2.241189,1.624317e-13,3.337795
87,dpAGWP,agwp_final,2050,50,N2O,1.027174e-11,1.039567e-11,1.048471e-11,1.239322e-13,1.206536,2.129762e-13,2.073420
88,dpAGWP,agwp_final,2050,100,N2O,1.555857e-11,1.555300e-11,1.557826e-11,-5.574941e-15,-0.035832,1.968732e-14,0.126537


In [3]:
df.to_csv(outdir / 'metric_only_ssp119_245_585_benchmark.csv', index=False)